# Setup lokalnego LLM-a

Ten notebook sprawdzi Twój sprzęt i pobierze model LLM do pracy na zajęciach.

**Na zajęciach pracujemy na `gemma4:e4b`** — lekki, szybki, świetny w function calling.

Notebook automatycznie wykryje co masz zainstalowane:
1. **LM Studio** (polecane) — jeśli masz zainstalowane, model trafi prosto do niego
2. **Ollama** — fallback, jeśli LM Studio nie jest zainstalowane

**Uruchom komórki po kolei (Shift+Enter).**

In [6]:
# === Krok 1: Sprawdzenie sprzętu ===
import platform
import subprocess
import sys
import os
from pathlib import Path

system = platform.system()

def get_ram_gb():
    try:
        if system == 'Darwin':
            out = subprocess.check_output(['sysctl', '-n', 'hw.memsize']).decode().strip()
            return int(out) / (1024 ** 3)
        elif system == 'Linux':
            with open('/proc/meminfo') as f:
                for line in f:
                    if 'MemTotal' in line:
                        return int(line.split()[1]) / (1024 ** 2)
        elif system == 'Windows':
            out = subprocess.check_output(
                ['wmic', 'computersystem', 'get', 'TotalPhysicalMemory'],
                shell=True
            ).decode()
            return int(out.strip().split()[-1]) / (1024 ** 3)
    except Exception as e:
        print(f'Nie udało się odczytać RAM: {e}')
    return None

def is_apple_silicon():
    if system != 'Darwin':
        return False
    try:
        chip = subprocess.check_output(['sysctl', '-n', 'machdep.cpu.brand_string']).decode().strip()
        return 'Apple' in chip
    except Exception:
        return False

def pick_model_for_ram(ram_gb):
    # Rekomendujemy największy model który się zmieści + gemma4:e4b na zajęcia
    if ram_gb >= 28:
        return 'qwen3.6:27b', 'Świetnie! Duże modele nie są problemem. Pobierz też gemma4:e4b — na zajęciach pracujemy na nim.'
    elif ram_gb >= 14:
        return 'qwen3.5:9b', 'Dobre — pójdzie sprawny 9B model. Pobierz też gemma4:e4b — na zajęciach pracujemy na nim.'
    elif ram_gb >= 6:
        return 'gemma4:e4b', 'OK — gemma4:e4b to świetny wybór. Lekki, szybki, świetny w function calling.'
    else:
        return None, 'Za mało RAM — skorzystaj z serwera prowadzącego na zajęciach.'

# Mapowanie: nazwa modelu → repo HuggingFace (MLX dla Apple Silicon, GGUF dla reszty)
HF_MODELS = {
    'gemma4:e4b': {
        'mlx': 'lmstudio-community/gemma-4-E4B-it-MLX-8bit',
        'gguf': 'lmstudio-community/gemma-4-E4B-it-GGUF',
    },
    'qwen3.5:9b': {
        'mlx': 'mlx-community/Qwen3.5-9B-4bit',
        'gguf': 'lmstudio-community/Qwen3.5-9B-GGUF',
    },
    'qwen3.6:27b': {
        'mlx': 'mlx-community/Qwen3.6-27B-4bit',
        'gguf': 'lmstudio-community/Qwen3.6-27B-GGUF',
    },
}

ram_gb = get_ram_gb()
apple_silicon = is_apple_silicon()
model, comment = pick_model_for_ram(ram_gb) if ram_gb else (None, 'Nie udało się odczytać RAM.')
fmt = 'mlx' if apple_silicon else 'gguf'

print(f'System:         {system}')
print(f'Apple Silicon:  {"tak" if apple_silicon else "nie"}')
print(f'RAM:            {ram_gb:.1f} GB' if ram_gb else 'RAM:            nieznany')
print(f'Format modeli:  {"MLX (szybsze na Macu)" if apple_silicon else "GGUF"}')
print(f'Rekomendacja:   {model or "brak"}')
print(f'                {comment}')

System:       Darwin
RAM:          32.0 GB
Rekomendowany model: qwen3.5:27b
Ocena:        Świetnie! Duże modele nie są problemem.


In [7]:
# === Krok 2: Co jest zainstalowane? ===
import requests

LMSTUDIO_DIR = Path.home() / '.lmstudio' / 'models'
HAS_LMSTUDIO = LMSTUDIO_DIR.exists()

def ollama_running():
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=3)
        return r.status_code == 200
    except Exception:
        return False

HAS_OLLAMA = ollama_running()

if HAS_LMSTUDIO:
    print(f'LM Studio wykryte! Folder modeli: {LMSTUDIO_DIR}')
    print('  → Modele zostaną pobrane do LM Studio.')
elif HAS_OLLAMA:
    print('LM Studio nie znalezione, ale Ollama działa!')
    print('  → Modele zostaną pobrane przez Ollamę.')
else:
    print('Nie znaleziono ani LM Studio ani Ollamy.')
    print()
    print('Zainstaluj jedno z nich:')
    print('  LM Studio (polecane): https://lmstudio.ai')
    print('  Ollama:               https://ollama.com')
    print()
    print('Po instalacji uruchom tę komórkę ponownie.')

Ollama już działa! Pomijam instalację.


In [8]:
# === Krok 3: Pobranie modeli ===
# Pobiera do LM Studio (jeśli zainstalowane) lub Ollamy (fallback)

if model is None:
    print('Za mało RAM — skorzystaj z serwera prowadzącego na zajęciach.')

elif HAS_LMSTUDIO:
    # ── LM Studio: pobieramy z HuggingFace ──
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        print('Instaluję huggingface_hub (potrzebne do pobrania modeli)...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
        from huggingface_hub import snapshot_download

    # Lista modeli do pobrania: zawsze gemma4:e4b + ewentualnie większy
    to_download = ['gemma4:e4b']
    if model != 'gemma4:e4b':
        to_download.append(model)

    for m in to_download:
        hf_info = HF_MODELS.get(m, {})
        repo = hf_info.get(fmt)
        if not repo:
            print(f'  Brak repo HuggingFace dla {m} ({fmt}) — pomiń.')
            continue

        target_dir = LMSTUDIO_DIR / repo.replace('/', os.sep)
        if target_dir.exists() and any(target_dir.glob('*.safetensors')):
            print(f'  {m} już pobrany → {target_dir.name}')
            continue

        print(f'  Pobieram {m} → {repo}')
        print(f'  (to może potrwać kilka minut...)')
        try:
            snapshot_download(
                repo_id=repo,
                local_dir=str(target_dir),
            )
            print(f'  Gotowe!\n')
        except Exception as e:
            print(f'  Błąd pobierania: {e}')
            print(f'  Spróbuj ręcznie w LM Studio: wyszukaj "{m.split(":")[0]}"\n')

elif HAS_OLLAMA:
    # ── Ollama: fallback ──
    to_download = ['gemma4:e4b']
    if model != 'gemma4:e4b':
        to_download.append(model)

    for m in to_download:
        r = requests.get('http://localhost:11434/api/tags', timeout=3)
        installed = [x['name'] for x in r.json().get('models', [])]
        if any(m.split(':')[0] in x for x in installed):
            print(f'  {m} już pobrany w Ollamie.')
            continue

        print(f'  Pobieram {m} przez Ollamę...')
        result = subprocess.run(['ollama', 'pull', m], capture_output=False)
        if result.returncode == 0:
            print(f'  {m} gotowy!\n')
        else:
            print(f'  Błąd pobierania {m}.\n')

else:
    print('Brak LM Studio i Ollamy — zainstaluj jedno z nich (patrz krok 2).')

Ollama już zainstalowana — nic do roboty.


In [ ]:
# === Krok 4: Weryfikacja ===

if HAS_LMSTUDIO:
    print('LM Studio — pobrane modele:')
    seen = set()
    for d in sorted(LMSTUDIO_DIR.iterdir()):
        if not d.is_dir():
            continue
        for model_dir in sorted(d.iterdir()):
            if not model_dir.is_dir():
                continue
            has_weights = any(model_dir.glob('*.safetensors')) or any(model_dir.glob('*.gguf'))
            if has_weights:
                size_gb = sum(f.stat().st_size for f in model_dir.iterdir() if f.is_file()) / (1024**3)
                print(f'  - {d.name}/{model_dir.name} ({size_gb:.1f} GB)')
    print()
    print('Otwórz LM Studio → załaduj model → Start Server → gotowe!')

elif HAS_OLLAMA:
    r = requests.get('http://localhost:11434/api/tags', timeout=3)
    models = [m['name'] for m in r.json().get('models', [])]
    if models:
        print('Ollama — pobrane modele:')
        for m in models:
            print(f'  - {m}')
    else:
        print('Ollama działa, ale brak modeli.')
    print()
    print('Ollama gotowa — możesz otworzyć notebook z zajęć!')

else:
    print('Brak LM Studio i Ollamy.')
    print('Na zajęciach skorzystaj z serwera prowadzącego (prowadzący poda adres).')

print()
print('Setup zakończony!')

Pobieram model qwen3.5:27b...
(może potrwać kilka minut — zależy od łącza)



pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling d4b8b4f4c350:   0% ▕                  ▏ 1.4 MB/ 17 GB                  pulling manifest 
pulling d4b8b4f4c350:   0% ▕                  ▏ 6.7 MB/ 17 GB                  pulling manifest 
pulling d4b8b4f4c350:   0% ▕                  ▏ 9.4 MB/ 17 GB                  pulling manifest 
pulling d4b8b4f4c350:   0% ▕                  ▏  14 MB/ 17 GB                  pulling manifest 
pulling d4b8b4f4c350:   0% ▕                  ▏  18 MB/ 17 GB                  pulling manifest 
pulling d4b8b4f4c350:   0% ▕                  ▏  21 MB/ 17 GB                  pulling manifest 
pulling d4b8b4f4c350:   0% ▕                  ▏  26 MB/ 17 GB                  pulling manifest 
pulling d4b8b4f4c350:   0% ▕                  ▏  31 MB/ 

Ollama działa. Dostępne modele:
  - gpt-oss:20b
  - antoniprzybylik/llama-pllum:8b
  - deepseek-r1:32b
  - SpeakLeash/bielik-11b-v2.3-instruct:Q4_K_M
  - deepseek-r1:1.5b
  - llama3.2-vision:11b
  - deepseek-coder-v2:16b
  - mistral:7b
  - deepseek-r1:8b
  - deepseek-r1:14b
  - llama3.2:latest

Setup zakończony — możesz otworzyć notebook z zajęć!
